# Lab 04: Your First Chain (LCEL)

**Goal:** Build your first chain using LCEL pipe syntax: `prompt | llm | parser`

**What you'll learn:**
- What LCEL (LangChain Expression Language) is
- The pipe operator `|` connects components
- `StrOutputParser` extracts text from the response
- Why chains are better than manual step-by-step calls

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.2:1b")

## Step 1: The Manual Way (Without Chains)

This works, but it's verbose and hard to reuse.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}"),
])

# Step by step:
messages = prompt.invoke({"question": "What is Flask?"})  # Step 1: format prompt
response = llm.invoke(messages)                             # Step 2: send to LLM
text = response.content                                     # Step 3: extract text

print(text)

## Step 2: The LCEL Way (One Chain, One Line)

The pipe `|` operator connects: prompt -> llm -> parser.
Each component's output automatically becomes the next one's input.

In [ ]:
chain = prompt | llm | StrOutputParser()

# Now invoke the entire chain in one call
result = chain.invoke({"question": "What is Flask?"})

print(result)
print(f"\nType: {type(result)}")  # It's a plain string now!

## Step 3: Understand the Data Flow

Let's trace what happens at each step.

In [ ]:
# Stage 1: Prompt takes a dict -> produces messages
stage1 = prompt.invoke({"question": "What is FastAPI?"})
print(f"After prompt:  {type(stage1).__name__} -> {len(stage1.to_messages())} messages")

# Stage 2: LLM takes messages -> produces AIMessage
stage2 = llm.invoke(stage1)
print(f"After LLM:     {type(stage2).__name__} -> '{stage2.content[:50]}...'")

# Stage 3: Parser takes AIMessage -> produces string
parser = StrOutputParser()
stage3 = parser.invoke(stage2)
print(f"After parser:  {type(stage3).__name__} -> '{stage3[:50]}...'")

# The chain does all three stages in one .invoke() call!

## Step 4: Reuse the Chain with Different Inputs

The chain is a reusable object. Pass different inputs each time.

In [ ]:
questions = [
    "What is Git in one sentence?",
    "What is an API in one sentence?",
    "What is a database in one sentence?",
]

for q in questions:
    answer = chain.invoke({"question": q})
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## TODO 1: Build a DevOps Explainer Chain

Create a chain: `devops_prompt | llm | StrOutputParser()`

The prompt should accept `{tool}` and `{audience}` variables.
Test it with different tools (Docker, Terraform, Ansible) and
audiences (beginner, DevOps engineer, CTO).

In [ ]:
# TODO: Build the DevOps explainer chain
# devops_prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a DevOps expert. Explain tools clearly for a {audience} audience. Keep it to 2 sentences."),
#     ("human", "Explain {tool}"),
# ])
#
# devops_chain = devops_prompt | llm | StrOutputParser()
#
# # Test with different tool + audience combos
# for tool, audience in [("Docker", "beginner"), ("Docker", "CTO"), ("Terraform", "DevOps engineer")]:
#     result = devops_chain.invoke({"tool": tool, "audience": audience})
#     print(f"{tool} (for {audience}): {result}\n")

## TODO 2: Build a Code Review Chain

Create a chain that takes `{code}` and `{language}` as input
and returns a brief code review (issues, suggestions).

This demonstrates a practical use case where the chain does
something more useful than simple Q&A.

In [ ]:
# TODO: Build the code review chain and test it
# review_prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a senior {language} developer. Review the code briefly: "
#      "mention 1 good thing and 1 improvement. Keep it to 3 sentences."),
#     ("human", "Review this code:\n```\n{code}\n```"),
# ])
#
# review_chain = review_prompt | llm | StrOutputParser()
#
# result = review_chain.invoke({
#     "language": "Python",
#     "code": "def get_user(id):\n    users = load_all_users()\n    for u in users:\n        if u['id'] == id:\n            return u\n    return None",
# })
# print(f"Review:\n{result}")

## Key Takeaways

- LCEL chain: `prompt | llm | parser`
- The pipe `|` passes output of each step to the next
- `StrOutputParser` extracts plain text from `AIMessage`
- Chains are reusable objects -- invoke with different inputs